In [13]:
import os
from pathlib import Path
import pandas as pd
import requests

# Load dataset
im3_df = pd.read_csv("Data/im3_atlas_cleaned.csv")

# Load environment variables
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()

api_key = os.environ.get("EIA_API_KEY")
BASE = "https://api.eia.gov/v2/"


def eia_get(route, params):
    rows, offset = [], 0
    while True:
        p = {
            **params,
            "api_key": api_key,
            "offset": offset,
            "length": 5000,
        }
        r = requests.get(BASE + route + "/data/", params=p)
        r.raise_for_status()
        resp = r.json()["response"]
        rows += resp["data"]
        offset += 5000
        if offset >= int(resp["total"]):
            break
    return pd.DataFrame(rows)


eia_params = {
    "frequency": "monthly",
    "data[0]": "price",
    "data[1]": "sales",
    "facets[sectorid][]": "COM",
    "sort[column]": "period",
    "sort[direction]": "desc",
}

eia_raw = eia_get("electricity/retail-sales", eia_params)

eia_clean = eia_raw[["period", "stateid", "price", "sales"]].copy()
eia_clean.rename(
    columns={
        "stateid": "state_abb",
        "price": "comm_elec_price_cents_kwh",
        "sales": "comm_elec_sales_mwh",
    },
    inplace=True,
)

eia_clean["comm_elec_price_cents_kwh"] = pd.to_numeric(
    eia_clean["comm_elec_price_cents_kwh"], errors="coerce"
)
eia_clean["comm_elec_sales_mwh"] = pd.to_numeric(
    eia_clean["comm_elec_sales_mwh"], errors="coerce"
)

latest_period = eia_clean["period"].max()
eia_latest = eia_clean[eia_clean["period"] == latest_period].copy()

im3_eia_merged = pd.merge(im3_df, eia_latest, on="state_abb", how="left")

print("Merged DataFrame Shape:", im3_eia_merged.shape)
print(im3_eia_merged.head())

HTTPError: 400 Client Error: Bad Request for url: https://api.eia.gov/v2/electricity/retail-sales/data/?frequency=monthly&data%5B0%5D=price&data%5B1%5D=sales&facets%5Bsectorid%5D%5B%5D=COM&sort%5Bcolumn%5D=period&sort%5Bdirection%5D=desc&api_key=5WFjMdvm3cCpgyPzFkOwcWn2bwYCsx6SdBLhBJ7l&offset=0&length=5000

loop

make a spatial join with column

In [14]:
import os
from pathlib import Path
import pandas as pd
import requests

# 1. Load your EIA API Key from the .env file
for line in Path(".env").read_text().splitlines():
    if "=" in line and not line.strip().startswith("#"):
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip()

api_key = os.environ.get("EIA_API_KEY")
BASE = "https://api.eia.gov/v2/"

# 2. Function to handle pagination and retrieve all rows
def eia_get(route, params):
    rows, offset = [], 0
    while True:
        p = {
            **params,
            "api_key": api_key,
            "offset": offset,
            "length": 5000,
        }
        r = requests.get(BASE + route + "/data/", params=p)
        r.raise_for_status()
        resp = r.json()["response"]
        rows += resp["data"]
        offset += 5000
        if offset >= int(resp["total"]):
            break
    return pd.DataFrame(rows)

# 3. Define parameters matching your retail sales query
eia_params = {
    "frequency": "monthly",
    "data[0]": "price",
    "data[1]": "sales",
    "facets[sectorid][]": "COM",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
}

# 4. Import the data straight into a Pandas DataFrame
eia_raw = eia_get("electricity/retail-sales", eia_params)

# Save a local copy so you don't have to fetch it every time
eia_raw.to_csv("Data/eia_retail_sales.csv", index=False)


Imported successfully! Total rows fetched: 19034


In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Clean and standardize EIA dataset columns
eia_clean = eia_raw[["period", "stateid", "price", "sales"]].copy()
eia_clean.rename(
    columns={
        "stateid": "state_abb",
        "price": "comm_elec_price_cents_kwh",
        "sales": "comm_elec_sales_mwh",
    },
    inplace=True,
)

# Convert string values to numeric numbers
eia_clean["comm_elec_price_cents_kwh"] = pd.to_numeric(
    eia_clean["comm_elec_price_cents_kwh"], errors="coerce"
)
eia_clean["comm_elec_sales_mwh"] = pd.to_numeric(
    eia_clean["comm_elec_sales_mwh"], errors="coerce"
)

# Filter out regional aggregate codes (US, ENC, etc.)
regional_aggregates = [
    "US", "ENC", "ESC", "MAT", "MTN", "NEW", "PACC", "PACN", "SAT", "WNC", "WSC"
]
eia_states = eia_clean[~eia_clean["state_abb"].isin(regional_aggregates)].copy()

# 2. Select the latest single month available to prevent row duplication
latest_period = eia_states["period"].max()
eia_latest = eia_states[eia_states["period"] == latest_period].copy()


# 3. Load IM3 dataset and create Spatial GeoDataFrame
im3_df = pd.read_csv("Data/im3_2025_atlas_cleaned.csv")

# Convert lat/lon coordinates into Spatial Geometry Points (WGS84 EPSG:4326)
im3_gdf = gpd.GeoDataFrame(
    im3_df,
    geometry=gpd.points_from_xy(im3_df["lon"], im3_df["lat"]),
    crs="EPSG:4326",
)


# 4. Perform Spatial/Attribute Join with EIA environmental grid metrics
# Some IM3 exports do not include state_abb, so backfill it from the richer IM3 table
if "state_abb" not in im3_gdf.columns:
    if "state_abb" not in im3_gdf.columns:
        im3["id"] = im3["id"].astype(str)
        im3_gdf["id"] = im3_gdf["id"].astype(str)
        im3_gdf = im3_gdf.merge(im3[["id", "state_abb"]], on="id", how="left")

im3_multimodal_gdf = im3_gdf.merge(eia_latest, on="state_abb", how="left")


# 5. Verify the merged multimodal dataset
print(f"Latest EIA Period Snapshot: {latest_period}")
print(f"Dataset Shape (Rows, Columns): {im3_multimodal_gdf.shape}")
print(f"Geometry Column Name: {im3_multimodal_gdf.geometry.name}")
print("\nFirst 5 Rows of Fused Multimodal Data:")
im3_multimodal_gdf[
    [
        "id",
        "state_abb",
        "sqft",
        "lat",
        "lon",
        "comm_elec_price_cents_kwh",
        "comm_elec_sales_mwh",
        "geometry",
    ]
].head()

ValueError: You are trying to merge on int64 and str columns for key 'id'. If you wish to proceed you should use pd.concat